In [0]:
from pyspark.sql.functions import *
from datetime import datetime


In [0]:
# Em execução via job: se a silver já está em sincronia com a bronze (nenhuma
# competência nova ou com contagem divergente), encerra imediatamente — a tarefa
# conclui com sucesso sem custo adicional. Interativamente, comente o exit para
# inspecionar as células seguintes.
TBL_BRONZE = "workspace.bronze.novo_caged_movimentacoes"
TBL_SILVER = "workspace.silver.novo_caged_movimentacoes"

tabela_existe = (
    spark.sql("SHOW TABLES IN workspace.silver LIKE 'novo_caged_movimentacoes'").count() > 0
)

if tabela_existe:
    pendentes = spark.sql(f"""
      SELECT count(*) AS n
      FROM (SELECT `_competencia_referencia` AS competencia, count(*) AS linhas_bronze
            FROM {TBL_BRONZE} GROUP BY 1) b
      LEFT JOIN (SELECT competencia_referencia AS competencia, count(*) AS linhas_silver
                 FROM {TBL_SILVER} GROUP BY 1) s
        ON b.competencia = s.competencia
      WHERE s.competencia IS NULL OR b.linhas_bronze != s.linhas_silver
    """).collect()[0]['n']
    if pendentes == 0:
        dbutils.notebook.exit("Silver já sincronizada com a bronze — nada a processar.")
    print(f"{pendentes} competência(s) pendente(s) para a silver.")
else:
    print("Tabela silver ainda não existe — primeira carga completa.")

## Conclusões — tratamentos necessários para a camada Silver

**Contexto:** 259.514.177 linhas, 68 competências (202101–202608) completas e coerentes (`competênciamov` = `competênciadec` = `_competencia_referencia` em 100% das linhas, todas no formato AAAAMM). Sem espaços excedentes em nenhuma coluna.

### 1. Tipagem (as 28 colunas de negócio estão como `string`)

| Colunas atuais | Tipo sugerido | Observação |
| --- | --- | --- |
| `competênciamov`, `competênciadec` | `DATE` | `to_date(concat(col,'01'),'yyyyMMdd')` |
| `idade`, `tamestabjan` | `INT` | |
| `horascontratuais` | `DECIMAL(5,2)` | min 0 / max 99 |
| `salário`, `valorsaláriofixo` | `DECIMAL(12,2)` | ver outliers no item 5 |
| `saldomovimentação`, `unidadesaláriocódigo` | `SMALLINT` | saldo é apenas -1/1 |
| demais códigos categóricos | `SMALLINT` ou manter `STRING` | manter código para join com `ref_*` |

### 2. Conversão de decimais — formatos mistos no Bronze

A maioria usa vírgula com 2 casas (`2219,80`), mas existem 3 desvios (~2% do total):

* `",00"` / `",01"` (parte inteira vazia): **1,36 mi** de linhas em `salário`/`valorsaláriofixo` — decidir política: `0.00` ou `NULL`
* Ponto como separador: `42.5`, `1977.04`, `1370.5` (~400 mil em `horascontratuais`)
* Artefatos de ponto flutuante: `1481.5601`, `51.330002`, `619.98999` — arredondar para 2 casas

Sugestão: `round(coalesce(try_cast(replace(col, ',', '.') AS DECIMAL(14,4)), try_cast(col AS DECIMAL(14,4))), 2)`

### 3. Nulos / vazios

* `valorsaláriofixo`: 1.440.011 (0,55%) · `salário`: 66.365 · `horascontratuais`: 78.342 · `idade`: 53.499
* Decidir se viram `NULL` tipado ou flag de qualidade (coluna `_indicador_*`)

### 4. Enriquecimento de-para com as `ref_*` (códigos sem match)

| Coluna | Código sem match | Linhas afetadas | Ação sugerida |
| --- | --- | --- | --- |
| `região` | `99` | 177.125 | não consta no dicionário (ref vai até `9` = Não identificado) — mapear para "Não identificado" ou manter sem descrição |
| `cbo2002ocupação` | 42 códigos (ex.: `423115`, `142360`) | 69.900 | manter código, descrição `NULL` |
| `subclasse` | `8630505` | 2.045 | idem |
| `município` | `510183` | 1.087 | idem |
| `tipoestabelecimento` | `-1` | 1 | valor inválido — tratar como `NULL`/"Não identificado" |

Colunas 100% aderentes: `uf` (inclui `99` = Ignorado), `seção`, `categoria`, `graudeinstrução`, `tipomovimentação`.

### 5. Outliers numéricos

* `salário` negativo: 1 linha · `valorsaláriofixo` negativo: 240 linhas
* Acima de 100 mil: 216.287 (`salário`) e 12.055 (`valorsaláriofixo`), incluindo o máximo de **1.000.000.000.000** (provável sentinela de valor ignorado)
* `idade` entre 14 e 100 (mediana 30) — faixa plausível, sem tratamento obrigatório

### 6. Duplicatas (sem chave natural na fonte)

Na competência 202608: 138.061 grupos de linhas idênticas (362.860 linhas, máximo de **211 repetições** da mesma linha; ~5,1% de linhas excedentes). Como o CAGED não traz identificador do trabalhador/estabelecimento, linhas idênticas **podem ser movimentos legítimos de perfis iguais** — recomendação: **não deduplicar** na silver e apenas documentar.

### 7. Renomeação de colunas

Nomes colados e acentuados (`competênciamov`, `cbo2002ocupação`, `graudeinstruação`) dificultam SQL — padronizar para `snake_case` sem acento na silver (ex.: `competencia_mov`, `uf`, `cbo2002_ocupacao`, `grau_instrucao`, `salario`, `valor_salario_fixo`).

### 8. O que NÃO precisa tratamento

* Competências: 100% coerentes, 68 períodos completos
* Sem espaços excedentes, sem valores vazios nas colunas de código
* Domínios categóricos íntegros (códigos `9`/`99`/`-1` = ignorado/inválido são raros e conhecidos)
* Metadados de ingestão (`_arquivo_origem`, `_caminho_origem`, `_data_ingestao`) prontos para linha de auditoria

## Construção da tabela Silver

Criação de `workspace.silver.novo_caged_movimentacoes` aplicando os tratamentos definidos acima:

* **Tipagem**: competências → `DATE`, códigos → `SMALLINT`/`INT`, valores → `DECIMAL`
* **Decimais**: normalização de vírgula/ponto, `",00"` → `0.00`, artefatos float arredondados para 2 casas (o parser em `DECIMAL(20,4)` acomoda a sentinela de 1 trilhão)
* **Renomeação**: colunas em `snake_case` sem acentos
* **Enriquecimento**: LEFT JOINs com as `ref_*` gerando colunas `*_descricao` (códigos sem match ficam com descrição `NULL` — ex.: `regiao=99`)
* **Nulos**: preservados como `NULL` tipado, sem imputação
* **Duplicatas**: mantidas, sem deduplicação (sem chave natural — item 6 das conclusões)
* **Clustering** liquid por `competencia_mov` (data da competência; o Delta só coleta stats das 32 primeiras colunas, por isso a chave não pode ficar no fim do schema)
* **Carga incremental**: a célula 6 só cria a tabela se ela ainda não existir; a célula 7 detecta automaticamente competências novas (ou com contagem divergente da bronze) e sobrescreve **somente** essas competências com `replaceWhere` em `competencia_referencia` — reprocessar um mês não afeta os demais

In [0]:
# 6) Mapeamento Bronze → Silver e criação da estrutura da tabela
TBL_BRONZE = "workspace.bronze.novo_caged_movimentacoes"
TBL_SILVER = "workspace.silver.novo_caged_movimentacoes"

def num(col, tipo):
    """Códigos/inteiros: try_cast preserva NULL de vazios e nulos."""
    return f"try_cast(`{col}` AS {tipo})"

def dec(col):
    """Decimais: aceita vírgula ('2219,80'), ponto ('42.5'), ',00' e artefatos float;
    arredonda para 2 casas. DECIMAL(20,4) acomoda o máximo de 1 trilhão."""
    return (f"round(coalesce(try_cast(replace(`{col}`, ',', '.') AS DECIMAL(20,4)), "
            f"try_cast(`{col}` AS DECIMAL(20,4))), 2)")

def comp(col):
    """Competência AAAAMM → DATE (dia 01)."""
    return f"to_date(concat(`{col}`, '01'), 'yyyyMMdd')"

# (nome_silver, tipo, expressão, comentário) — ordem das colunas da silver
colunas = [
    ('competencia_mov', 'DATE', comp('competênciamov'), 'Competência da movimentação (mês de referência)'),
    ('competencia_dec', 'DATE', comp('competênciadec'), 'Competência declarada'),
    ('regiao', 'SMALLINT', num('região', 'SMALLINT'), 'Código da região'),
    ('uf', 'SMALLINT', num('uf', 'SMALLINT'), 'Código da UF'),
    ('municipio', 'INT', num('município', 'INT'), 'Código do município'),
    ('secao', 'STRING', 'trim(`seção`)', 'Código da seção CNAE'),
    ('subclasse', 'INT', num('subclasse', 'INT'), 'Código da subclasse CNAE'),
    ('saldo_movimentacao', 'SMALLINT', num('saldomovimentação', 'SMALLINT'), 'Saldo da movimentação (1 = admissão, -1 = desligamento)'),
    ('cbo2002_ocupacao', 'INT', num('cbo2002ocupação', 'INT'), 'Código CBO 2002 da ocupação'),
    ('categoria', 'SMALLINT', num('categoria', 'SMALLINT'), 'Código da categoria'),
    ('grau_instrucao', 'SMALLINT', num('graudeinstrução', 'SMALLINT'), 'Código do grau de instrução'),
    ('idade', 'SMALLINT', num('idade', 'SMALLINT'), 'Idade do trabalhador (14 a 100)'),
    ('horas_contratuais', 'DECIMAL(5,2)', dec('horascontratuais'), 'Horas contratuais (0 a 99)'),
    ('raca_cor', 'SMALLINT', num('raçacor', 'SMALLINT'), 'Código de raça/cor'),
    ('sexo', 'SMALLINT', num('sexo', 'SMALLINT'), 'Sexo (1 = masculino, 3 = feminino, 9 = ignorado)'),
    ('tipo_empregador', 'SMALLINT', num('tipoempregador', 'SMALLINT'), 'Código do tipo de empregador'),
    ('tipo_estabelecimento', 'SMALLINT', num('tipoestabelecimento', 'SMALLINT'), 'Código do tipo de estabelecimento'),
    ('tipo_movimentacao', 'SMALLINT', num('tipomovimentação', 'SMALLINT'), 'Código do tipo de movimentação'),
    ('tipo_deficiencia', 'SMALLINT', num('tipodedeficiência', 'SMALLINT'), 'Código do tipo de deficiência'),
    ('ind_trab_intermitente', 'SMALLINT', num('indtrabintermitente', 'SMALLINT'), 'Indicador de trabalho intermitente'),
    ('ind_trab_parcial', 'SMALLINT', num('indtrabparcial', 'SMALLINT'), 'Indicador de trabalho parcial'),
    ('salario', 'DECIMAL(18,2)', dec('salário'), 'Salário na unidade indicada por unidade_salario_codigo'),
    ('tam_estab_jan', 'SMALLINT', num('tamestabjan', 'SMALLINT'), 'Tamanho do estabelecimento em janeiro (99 = ignorado)'),
    ('indicador_aprendiz', 'SMALLINT', num('indicadoraprendiz', 'SMALLINT'), 'Indicador de aprendiz'),
    ('origem_informacao', 'SMALLINT', num('origemdainformação', 'SMALLINT'), 'Origem da informação'),
    ('indicador_fora_prazo', 'SMALLINT', num('indicadordeforadoprazo', 'SMALLINT'), 'Indicador de movimentação fora do prazo'),
    ('unidade_salario_codigo', 'SMALLINT', num('unidadesaláriocódigo', 'SMALLINT'), 'Código da unidade do salário'),
    ('valor_salario_fixo', 'DECIMAL(18,2)', dec('valorsaláriofixo'), 'Valor do salário fixo'),
]

# Enriquecimento de-para: coluna silver → (coluna bronze, tabela de referência)
depara = {
    'regiao': ('região', 'ref_regiao'),
    'uf': ('uf', 'ref_uf'),
    'municipio': ('município', 'ref_municipio'),
    'secao': ('seção', 'ref_secao'),
    'subclasse': ('subclasse', 'ref_subclasse'),
    'cbo2002_ocupacao': ('cbo2002ocupação', 'ref_cbo_ocupacao'),
    'categoria': ('categoria', 'ref_categoria'),
    'grau_instrucao': ('graudeinstrução', 'ref_grau_instrucao'),
    'tipo_movimentacao': ('tipomovimentação', 'ref_tipo_movimentacao'),
    'tipo_estabelecimento': ('tipoestabelecimento', 'ref_tipo_estabelecimento'),
}

# Colunas de auditoria herdadas da bronze
meta = [
    ('competencia_referencia', 'STRING', '`_competencia_referencia`', 'Competência da carga (AAAAMM)'),
    ('arquivo_origem', 'STRING', '`_arquivo_origem`', 'Arquivo .7z de origem'),
    ('caminho_origem', 'STRING', '`_caminho_origem`', 'Caminho do arquivo no volume'),
    ('data_ingestao', 'TIMESTAMP', '`_data_ingestao`', 'Data/hora da ingestão na bronze'),
]

# Monta SELECT (com descrições intercaladas), DDL e JOINs na mesma ordem
sel_exprs, ddl_cols, joins = [], [], []
for nome, tipo, expr, coment in colunas:
    sel_exprs.append(f"{expr} AS {nome}")
    ddl_cols.append((nome, tipo, coment))
    if nome in depara:
        bronze_col, ref = depara[nome]
        a = f"d_{nome}"
        sel_exprs.append(f"{a}.descricao AS {nome}_descricao")
        ddl_cols.append((f"{nome}_descricao", 'STRING', f'Descrição de {nome} (de-para: {ref})'))
        joins.append(f"LEFT JOIN workspace.bronze.{ref} {a} ON b.`{bronze_col}` = {a}.codigo")
for nome, tipo, expr, coment in meta:
    sel_exprs.append(f"{expr} AS {nome}")
    ddl_cols.append((nome, tipo, coment))

sql_select = "SELECT " + ",\n       ".join(sel_exprs) + f"\nFROM {TBL_BRONZE} b\n" + "\n".join(joins)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
tabela_existe = (
    spark.sql("SHOW TABLES IN workspace.silver LIKE 'novo_caged_movimentacoes'").count() > 0
)
if tabela_existe:
    print(f"Tabela {TBL_SILVER} já existe — estrutura preservada (carga será incremental).")
else:
    ddl = ",\n  ".join(f"`{n}` {t} COMMENT '{c}'" for n, t, c in ddl_cols)
    spark.sql(f"""
      CREATE TABLE {TBL_SILVER} (
        {ddl}
      )
      USING DELTA
      CLUSTER BY (competencia_mov)
    """)
    print(f"Estrutura criada: {TBL_SILVER} — {len(ddl_cols)} colunas, clustering liquid por competencia_mov")

In [0]:
from pyspark.sql import functions as F

# 7) Carga incremental por competência: processa apenas competências novas na silver
#    (ausentes ou com contagem divergente da bronze). Cada competência é sobrescrita
#    isoladamente (replaceWhere em competencia_referencia), o que torna a execução
#    idempotente — reprocessar um mês não afeta os demais.
# Reprocessamento manual (mesmo sem divergência detectada) — ex.: ['202601']
competencias_forcar = []

print("Detectando competências pendentes (bronze × silver)...")
pendentes = spark.sql(f"""
  SELECT b.competencia, b.linhas_bronze, coalesce(s.linhas_silver, 0) AS linhas_silver
  FROM (SELECT `_competencia_referencia` AS competencia, count(*) AS linhas_bronze
        FROM {TBL_BRONZE} GROUP BY 1) b
  LEFT JOIN (SELECT competencia_referencia AS competencia, count(*) AS linhas_silver
             FROM {TBL_SILVER} GROUP BY 1) s
    ON b.competencia = s.competencia
  WHERE s.competencia IS NULL OR b.linhas_bronze != s.linhas_silver
  ORDER BY b.competencia
""").collect()
for r in pendentes:
    print(f"  pendente: {r['competencia']} — bronze {r['linhas_bronze']:,} × silver {r['linhas_silver']:,}")

a_processar = sorted(set(competencias_forcar) | {r['competencia'] for r in pendentes})

if not a_processar:
    print("Nenhuma competência pendente — silver em sincronia com a bronze.")
else:
    print(f"\n{len(a_processar)} competência(s) a processar: {', '.join(a_processar)}")
    for i, comp in enumerate(a_processar, 1):
        df_comp = spark.sql(f"{sql_select}\nWHERE b.`_competencia_referencia` = '{comp}'")
        n_bronze = df_comp.count()
        assert n_bronze > 0, f"{comp}: nenhum registro na bronze."
        print(f"[{i}/{len(a_processar)}] {comp}: {n_bronze:,} linhas — gravando...")
        (
            df_comp.writeTo(TBL_SILVER)
            .using("delta")
            .overwrite(F.col("competencia_referencia") == comp)
        )
        n_silver = spark.sql(
            f"SELECT count(*) AS n FROM {TBL_SILVER} WHERE competencia_referencia = '{comp}'"
        ).collect()[0]['n']
        assert n_silver == n_bronze, f"{comp}: silver ({n_silver:,}) != bronze ({n_bronze:,})"
        print(f"         {comp}: OK — {n_silver:,} linhas na silver")
    print("\nCarga incremental concluída.")

In [0]:
# 8) Validação da Silver: volumetria, de-para, nulos, faixas e amostra
# Valores esperados: 259.514.177 linhas, 68 competências, sem desc. região = 177.125,
# sem desc. CBO = 69.900, salário zero >= 1.355.040 (linhas ',00' da bronze)
display(spark.sql(f"""
  SELECT count(*) AS total_linhas,
         count(DISTINCT competencia_referencia) AS competencias,
         min(competencia_mov) AS primeira_competencia,
         max(competencia_mov) AS ultima_competencia,
         sum(CASE WHEN regiao_descricao IS NULL THEN 1 ELSE 0 END) AS linhas_sem_desc_regiao,
         sum(CASE WHEN cbo2002_ocupacao_descricao IS NULL THEN 1 ELSE 0 END) AS linhas_sem_desc_cbo,
         sum(CASE WHEN tipo_estabelecimento_descricao IS NULL THEN 1 ELSE 0 END) AS linhas_sem_desc_tipo_estab,
         sum(CASE WHEN salario IS NULL THEN 1 ELSE 0 END) AS salario_nulo,
         sum(CASE WHEN salario = 0 THEN 1 ELSE 0 END) AS salario_zero,
         sum(CASE WHEN valor_salario_fixo IS NULL THEN 1 ELSE 0 END) AS valor_fixo_nulo,
         min(salario) AS salario_min, max(salario) AS salario_max,
         min(idade) AS idade_min, max(idade) AS idade_max
  FROM {TBL_SILVER}"""))

display(spark.sql(f"SELECT * FROM {TBL_SILVER} LIMIT 5"))